# 06 — Heuristica Lexical Ponderada (M5)

**Objetivo:** Avaliar o classificador heuristico (195 termos, 7 temas, 4 niveis de sinal) nos modos estrito e leniente sobre o split de validacao.

**Nota:** A heuristica nao requer treino. A avaliacao aplica o scoring a cada texto e classifica por limiares fixos.

**Dependencias:** Notebook 01 executado (splits em `artifacts/splits/`).

In [ ]:
import pandas as pd
import numpy as np

from economy_classifier.heuristics import (
    build_default_catalog, score_text, classify_score,
    MARKET_SCORE_THRESHOLD, AMBIGUOUS_SCORE_THRESHOLD,
)
from economy_classifier.evaluation import compute_binary_metrics, compute_roc_auc
from economy_classifier.project import (
    SPLITS_DIR, create_run_directory, build_run_metadata, persist_run_artifacts,
)
from economy_classifier.visualization import (
    configure_style, plot_confusion_matrix, plot_roc_curve, save_figure,
)

In [ ]:
configure_style()

val = pd.read_parquet(SPLITS_DIR / 'val.parquet')
catalog = build_default_catalog()

print(f'Validacao: {len(val):,} linhas')
print(f'Catalogo:  {len(catalog)} termos')
print(f'Limiar mercado (estrito): {MARKET_SCORE_THRESHOLD}')
print(f'Limiar ambiguo:          {AMBIGUOUS_SCORE_THRESHOLD}')

## Scoring

In [ ]:
import time

t0 = time.perf_counter()
results = []
for idx, row in val.iterrows():
    text = str(row['text']) if pd.notna(row['text']) else ''
    r = score_text(text, catalog)
    results.append({
        'index': idx,
        'y_true': row['label'],
        'score': r['score'],
        'classification': r['classification'],
    })
    if len(results) % 5000 == 0:
        print(f'  {len(results):,}/{len(val):,}...')

scoring_time = time.perf_counter() - t0
print(f'Concluido em {scoring_time:.1f}s')

results_df = pd.DataFrame(results)

## Distribuicao das bandas

In [ ]:
band_dist = results_df['classification'].value_counts()
print('Distribuicao das classificacoes heuristicas:')
for band, count in band_dist.items():
    print(f'  {band:>10}: {count:>6,} ({count/len(val)*100:.1f}%)')

## Avaliacao — Modo estrito

Somente a banda `mercado` conta como positivo.

In [ ]:
y_true = results_df['y_true'].values
y_pred_strict = (results_df['classification'] == 'mercado').astype(int).values

raw_scores = results_df['score'].values
s_min, s_max = raw_scores.min(), raw_scores.max()
y_score_norm = (raw_scores - s_min) / (s_max - s_min) if s_max > s_min else np.zeros_like(raw_scores)

metrics_strict = compute_binary_metrics(y_true, y_pred_strict)
metrics_strict['auc_roc'] = round(compute_roc_auc(y_true, y_score_norm), 4)

print('Modo estrito:')
for k, v in metrics_strict.items():
    print(f'  {k:<12}: {v:.4f}')

## Avaliacao — Modo leniente

Bandas `mercado` + `ambiguo` contam como positivo (analise de sensibilidade).

In [ ]:
y_pred_lenient = results_df['classification'].isin(['mercado', 'ambiguo']).astype(int).values

metrics_lenient = compute_binary_metrics(y_true, y_pred_lenient)
metrics_lenient['auc_roc'] = round(compute_roc_auc(y_true, y_score_norm), 4)

print('Modo leniente:')
for k, v in metrics_lenient.items():
    print(f'  {k:<12}: {v:.4f}')

## Figuras

In [ ]:
run_dir_strict = create_run_directory('heuristic-eval', 'strict')

fig_cm = plot_confusion_matrix(y_true, y_pred_strict, title='M5 Heuristica (estrito) — Matriz de Confusao (val)')
save_figure(fig_cm, run_dir_strict / 'figures', 'heuristic_strict_confusion_matrix')

fig_roc = plot_roc_curve(y_true, y_score_norm, title='M5 Heuristica — Curva ROC (val)')
save_figure(fig_roc, run_dir_strict / 'figures', 'heuristic_roc_curve')

## Persistencia

In [ ]:
# Estrito
preds_strict = pd.DataFrame({
    'index': results_df['index'].tolist(),
    'y_true': y_true.tolist(),
    'y_pred': y_pred_strict.tolist(),
    'y_score': np.round(y_score_norm, 4).tolist(),
    'method': 'heuristic_strict',
})
preds_strict.to_csv(run_dir_strict / 'predictions.csv', index=False)

meta_strict = build_run_metadata(
    run_dir=run_dir_strict, stage='heuristic-eval',
    parameters={'mode': 'strict', 'catalog_size': len(catalog), 'threshold': MARKET_SCORE_THRESHOLD},
    inputs={'val_rows': len(val)},
    outputs={'predictions': str(run_dir_strict / 'predictions.csv')},
    summary={'method': 'heuristic_strict', **metrics_strict},
    timing={'scoring_seconds': round(scoring_time, 2)},
)
persist_run_artifacts(run_dir=run_dir_strict, metadata=meta_strict, metrics=metrics_strict)

# Leniente
run_dir_lenient = create_run_directory('heuristic-eval', 'lenient')
preds_lenient = pd.DataFrame({
    'index': results_df['index'].tolist(),
    'y_true': y_true.tolist(),
    'y_pred': y_pred_lenient.tolist(),
    'y_score': np.round(y_score_norm, 4).tolist(),
    'method': 'heuristic_lenient',
})
preds_lenient.to_csv(run_dir_lenient / 'predictions.csv', index=False)

meta_lenient = build_run_metadata(
    run_dir=run_dir_lenient, stage='heuristic-eval',
    parameters={'mode': 'lenient', 'catalog_size': len(catalog)},
    inputs={'val_rows': len(val)},
    outputs={'predictions': str(run_dir_lenient / 'predictions.csv')},
    summary={'method': 'heuristic_lenient', **metrics_lenient},
    timing={'scoring_seconds': round(scoring_time, 2)},
)
persist_run_artifacts(run_dir=run_dir_lenient, metadata=meta_lenient, metrics=metrics_lenient)

print(f'Artefatos salvos em:')
print(f'  Estrito:  {run_dir_strict}')
print(f'  Leniente: {run_dir_lenient}')